# 📋 Python `logging` — Интерактивный справочник

> **Каждая ячейка самодостаточна.** Запускай по одной — сразу видишь результат.

| # | Раздел |
|---|--------|
| 0 | Настройка окружения (`fresh_logger`) |
| 1 | Уровни логирования |
| 2 | Formatter — формат вывода |
| 3 | Handlers — StreamHandler, FileHandler |
| 4 | basicConfig — быстрый старт |
| 5 | Иерархия логгеров и propagate |
| 6 | Логирование исключений |
| 7 | dictConfig — продвинутая настройка |

---
## 0. Настройка окружения

`fresh_logger()` — вспомогательная функция, которая создаёт **изолированный логгер** для каждого примера,
чтобы ячейки не влияли друг на друга.

In [ ]:
import logging
import logging.config
import sys

def fresh_logger(
    name: str,
    level: int = logging.DEBUG,
    fmt: str | None = None,
) -> logging.Logger:
    """Создаёт изолированный логгер для демонстрации."""
    logger = logging.getLogger(name)
    logger.handlers.clear()       # сброс при повторном запуске ячейки
    logger.setLevel(level)
    fmt = fmt or "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(fmt, datefmt="%H:%M:%S"))
    logger.addHandler(handler)
    logger.propagate = False      # не пробрасывать в root-логгер
    return logger

print('✅ Готово — можно запускать остальные ячейки')

---
## 1. Уровни логирования

| Уровень    | Значение | По умолчанию |
|------------|:--------:|:------------:|
| `DEBUG`    | 10       | ❌           |
| `INFO`     | 20       | ❌           |
| `WARNING`  | 30       | ✅           |
| `ERROR`    | 40       | ✅           |
| `CRITICAL` | 50       | ✅           |

Выводятся все сообщения с уровнем **≥** установленному.

In [ ]:
# Все уровни при level=DEBUG
logger = fresh_logger('demo.levels')

logger.debug('DEBUG    — подробная отладка')
logger.info('INFO     — штатная работа')
logger.warning('WARNING  — потенциальная проблема')
logger.error('ERROR    — функциональность нарушена')
logger.critical('CRITICAL — критический сбой')

In [ ]:
# Фильтрация: при WARNING — DEBUG и INFO не выведутся
logger = fresh_logger('demo.levels.filter', level=logging.WARNING)

logger.debug('Не выведется (DEBUG < WARNING)')   # ← скрыто
logger.info('Не выведется (INFO < WARNING)')     # ← скрыто
logger.warning('Выведется — уровень WARNING')
logger.error('Выведется — уровень ERROR')

---
## 2. Formatter — формат вывода

| Поле              | Описание                        |
|-------------------|---------------------------------|
| `%(asctime)s`     | Время создания записи           |
| `%(levelname)s`   | Уровень (`DEBUG`, `INFO`, ...)  |
| `%(name)s`        | Имя логгера                     |
| `%(message)s`     | Текст сообщения                 |
| `%(filename)s`    | Имя файла                       |
| `%(funcName)s`    | Имя функции                     |
| `%(lineno)d`      | Номер строки                    |
| `%(process)d`     | PID процесса                    |

**Трюк выравнивания:** `%(levelname)-8s` — минимум 8 символов, выравнивание влево.

In [ ]:
# Минимальный формат
logger = fresh_logger('demo.fmt.min', fmt='%(levelname)s: %(message)s')
logger.warning('Минимальный формат')
logger.error('Только уровень и сообщение')

In [ ]:
# Стандартный формат с выравниванием
fmt = '%(asctime)s | %(levelname)-8s | %(name)s:%(lineno)d | %(message)s'
logger = fresh_logger('demo.fmt.standard', fmt=fmt)
logger.info('Стандартный формат')
logger.error('С именем логгера и номером строки')

In [ ]:
# Кастомный формат даты через datefmt
logger = logging.getLogger('demo.fmt.date')
logger.handlers.clear()
logger.setLevel(logging.DEBUG)
logger.propagate = False

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter(
    fmt='[%(asctime)s] %(levelname)s — %(message)s',
    datefmt='%d.%m.%Y %H:%M:%S',   # ДД.ММ.ГГГГ ЧЧ:ММ:СС
))
logger.addHandler(handler)

logger.info('Формат даты: ДД.ММ.ГГГГ ЧЧ:ММ:СС')

---
## 3. Handlers — куда писать

| Handler                    | Назначение                        |
|----------------------------|-----------------------------------|
| `StreamHandler`            | Вывод в поток (stdout / stderr)   |
| `FileHandler`              | Запись в файл                     |
| `RotatingFileHandler`      | Файл с ротацией по размеру        |
| `TimedRotatingFileHandler` | Файл с ротацией по времени        |

Один логгер может иметь **несколько хендлеров** с разными уровнями.

In [ ]:
# StreamHandler — вывод в stdout
logger = logging.getLogger('demo.stream')
logger.handlers.clear()
logger.setLevel(logging.DEBUG)
logger.propagate = False

sh = logging.StreamHandler(sys.stdout)
sh.setLevel(logging.DEBUG)
sh.setFormatter(logging.Formatter('[STREAM] %(levelname)-8s | %(message)s'))
logger.addHandler(sh)

logger.info('Это идёт в stdout')
logger.error('Это тоже')

In [ ]:
# Два хендлера: консоль (WARNING+) и файл (DEBUG+)
import tempfile, os

log_file = os.path.join(tempfile.gettempdir(), 'demo_logging.log')

logger = logging.getLogger('demo.multi')
logger.handlers.clear()
logger.setLevel(logging.DEBUG)
logger.propagate = False

fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%H:%M:%S')

# Консоль — WARNING и выше
ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.WARNING)
ch.setFormatter(fmt)

# Файл — всё с DEBUG
fh = logging.FileHandler(log_file, encoding='utf-8')
fh.setLevel(logging.DEBUG)
fh.setFormatter(fmt)

logger.addHandler(ch)
logger.addHandler(fh)

logger.debug('DEBUG  → только в файл')      # в ячейке не появится
logger.info('INFO   → только в файл')       # в ячейке не появится
logger.warning('WARNING → консоль + файл')
logger.error('ERROR   → консоль + файл')

print(f'\n📄 Содержимое файла {log_file}:')
with open(log_file, encoding='utf-8') as f:
    print(f.read())

---
## 4. basicConfig — быстрый старт

`logging.basicConfig()` настраивает **root-логгер** одной командой.

> ⚠️ Работает **один раз**. Повторный вызов игнорируется, если хендлеры уже добавлены.  
> Используй `force=True` для принудительного сброса.

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout,  # в Jupyter лучше stdout (не stderr)
    force=True,         # принудительный сброс предыдущих хендлеров
)

logging.debug('Через root-логгер')
logging.info('INFO через root')
logging.warning('WARNING через root')

---
## 5. Иерархия логгеров и `propagate`

```
root
└── app                ← хендлер здесь
    ├── app.db         ← propagate=True → сообщения идут в app
    │   └── app.db.query
    └── app.api
```

`propagate=True` (по умолчанию) — сообщение поднимается вверх по дереву.

In [ ]:
# Сброс логгеров перед демонстрацией
for name in ['app', 'app.db', 'app.db.query']:
    logging.getLogger(name).handlers.clear()
    logging.getLogger(name).propagate = True

# Хендлер только на parent
parent = logging.getLogger('app')
parent.setLevel(logging.DEBUG)
parent.propagate = False
ch = logging.StreamHandler(sys.stdout)
ch.setFormatter(logging.Formatter('[%(name)s] %(levelname)s: %(message)s'))
parent.addHandler(ch)

# Дочерние логгеры — без хендлеров
child = logging.getLogger('app.db')
grandchild = logging.getLogger('app.db.query')

parent.info('Из app — напрямую')
child.warning('Из app.db → пробросится в app')
grandchild.error('Из app.db.query → app.db → app')

In [ ]:
# propagate=False — остановить пробрасывание
child = logging.getLogger('app.db')
child.propagate = False

grandchild = logging.getLogger('app.db.query')
grandchild.error('Не дойдёт до app (propagate=False у app.db, нет своих хендлеров)')

print('(строка выше не вывелась — propagate=False и нет хендлеров у app.db)')

---
## 6. Логирование исключений

| Вызов                              | Уровень | Traceback |
|------------------------------------|:-------:|:---------:|
| `logger.exception(msg)`            | ERROR   | ✅ авто   |
| `logger.error(msg, exc_info=True)` | ERROR   | ✅        |
| `logger.error(msg)`                | ERROR   | ❌        |

In [ ]:
logger = fresh_logger('demo.exc')

try:
    result = 1 / 0
except ZeroDivisionError:
    logger.exception('Деление на ноль — exception() добавляет traceback автоматически')

In [ ]:
logger = fresh_logger('demo.exc.info')

try:
    int('не число')
except ValueError:
    logger.error('Явный exc_info=True', exc_info=True)     # то же что exception()
    logger.warning('Без traceback', exc_info=False)         # только сообщение

---
## 7. dictConfig — продвинутая настройка

Позволяет описать **всю конфигурацию логирования** в одном словаре.  
Используется в продакшн-проектах (FastAPI, Django и т.д.).

In [ ]:
LOGGING_CONFIG = {
    'version': 1,
    'disable_existing_loggers': False,
    'formatters': {
        'standard': {
            'format': '%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
            'datefmt': '%H:%M:%S',
        },
        'simple': {
            'format': '%(levelname)s: %(message)s',
        },
    },
    'handlers': {
        'console': {
            'class': 'logging.StreamHandler',
            'level': 'DEBUG',
            'formatter': 'standard',
            'stream': 'ext://sys.stdout',
        },
    },
    'loggers': {
        'my_app': {
            'level': 'DEBUG',
            'handlers': ['console'],
            'propagate': False,
        },
    },
}

logging.config.dictConfig(LOGGING_CONFIG)

logger = logging.getLogger('my_app')
logger.debug('Настроено через dictConfig')
logger.info('Удобно для продакшн-проектов')
logger.warning('Вся конфигурация в одном месте')